## $z(t) = - t - Ae^{-2\kappa t} + B$

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.special import gamma

np.set_printoptions(precision=2, suppress=True)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# discretization
N = 1000        # time samples
M = 10        # p, omega modes (natural numbers)

EPOCHS = 100     # keep small; this is heavy
BATCH_SIZE = 1  
LR = 1e-4

In [3]:
p_grid     = np.arange(1, M + 1)      # p = 1,2,...,1000
omega_grid = np.arange(1, M + 1)      # ω = 1,2,...,1000

In [4]:
def sample_parameters():
    A = np.random.uniform(0.1, 1.0)
    B = np.random.uniform(0.1, 1.0)
    kappa = np.random.uniform(0.5, 1.5)
    return A, B, kappa

In [5]:
def mirror_trajectory(t, A, B, kappa):
    return -t - A * np.exp(-2 * kappa * t) + B

In [6]:
def compute_alpha_beta(A, B, kappa, p_grid=p_grid, omega_grid=omega_grid):
    """
    Computes alpha_{pω} and beta_{pω} exactly as in the paper.

    p_grid, omega_grid : 1D numpy arrays of natural numbers
    Returns:
        alpha, beta : complex numpy arrays of shape (len(p_grid), len(omega_grid))
    """

    # reshape for broadcasting
    p = p_grid[:, None].astype(np.complex128)
    w = omega_grid[None, :].astype(np.complex128)

    # D parameter
    D = (1.0 / kappa) * np.log(A) - B

    # common factors
    prefactor = 1j / (2.0 * np.pi) * 1.0 / (p * w)

    gamma_factor = gamma(1.0 + 1j * p / kappa)
    power_factor = np.exp(-1j * (p / kappa) * np.log(w))
    phase_p = np.exp(-1j * p * D)

    # alpha_{pω}
    alpha = (
        -prefactor
        * np.exp(+np.pi * p / (2.0 * kappa))
        * phase_p
        * np.exp(+1j * w * B)
        * power_factor
        * gamma_factor
    )

    # beta_{pω}
    beta = (
        +prefactor
        * np.exp(-np.pi * p / (2.0 * kappa))
        * phase_p
        * np.exp(-1j * w * B)
        * power_factor
        * gamma_factor
    )

    return alpha, beta

In [7]:
class MirrorDataset(Dataset):
    def __init__(self, n_samples):
        self.t = np.linspace(50, 100, N)
        self.samples = []

        for _ in range(n_samples):
            A, B, kappa = sample_parameters()
            z = mirror_trajectory(self.t, A, B, kappa)

            alpha, beta = compute_alpha_beta(A, B, kappa)

            self.samples.append((
                torch.tensor(z, dtype=torch.complex64),
                torch.tensor(alpha),
                torch.tensor(beta)
            ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        return self.samples[i]

In [8]:
class ComplexTanh(nn.Module):
    def forward(self, z):
        return torch.tanh(z.real) + 1j * torch.tanh(z.imag)

class ComplexMatrixLinear(nn.Module):
    def __init__(self, M):
        super().__init__()
        self.linear = nn.Linear(
            M * M, M * M, bias=True, dtype=torch.complex64
        )

    def forward(self, x):
        # x: (batch, M, M)
        b = x.shape[0]
        x = x.view(b, M * M)
        x = self.linear(x)
        return x.view(b, M, M)

In [9]:
class MatrixNet(nn.Module):
    def __init__(self, depth=2):
        super().__init__()

        self.embed = nn.Linear(N, M * M, dtype=torch.complex64)

        self.layers = nn.ModuleList(
            [ComplexMatrixLinear(M) for _ in range(depth)]
        )

        self.act = ComplexTanh()

    def forward(self, z):
        # z: (batch, N)
        x = self.embed(z).view(-1, M, M)

        for layer in self.layers:
            x = self.act(layer(x))

        return x

In [10]:
dataset = MirrorDataset(n_samples=100)
loader = DataLoader(dataset, batch_size=1, shuffle=True)

alpha_net = MatrixNet().to(device)
beta_net  = MatrixNet().to(device)

opt_a = torch.optim.Adam(alpha_net.parameters(), lr=LR)
opt_b = torch.optim.Adam(beta_net.parameters(), lr=LR)

def complex_mse(x, y):
    diff = x - y
    return (diff.real**2 + diff.imag**2).mean()

In [11]:
for epoch in range(EPOCHS):
    for z, alpha_t, beta_t in loader:
        z = z.to(device)
        alpha_t = alpha_t.to(device)
        beta_t = beta_t.to(device)

        opt_a.zero_grad()
        loss_a = complex_mse(alpha_net(z), alpha_t)
        loss_a.backward()
        opt_a.step()

        opt_b.zero_grad()
        loss_b = complex_mse(beta_net(z), beta_t)
        loss_b.backward()
        opt_b.step()

    if ((epoch+1) % 10 == 0):
        print(f"Epoch {epoch+1}: Loss for 'α': {loss_a.item():.6f} | Loss for 'β': {loss_b.item():.6f}")

Epoch 10: Loss for 'α': 0.011443 | Loss for 'β': 0.000002
Epoch 20: Loss for 'α': 0.008541 | Loss for 'β': 0.000006
Epoch 30: Loss for 'α': 0.011778 | Loss for 'β': 0.000012
Epoch 40: Loss for 'α': 0.005907 | Loss for 'β': 0.000005
Epoch 50: Loss for 'α': 0.008525 | Loss for 'β': 0.000002
Epoch 60: Loss for 'α': 0.006212 | Loss for 'β': 0.000003
Epoch 70: Loss for 'α': 0.006503 | Loss for 'β': 0.000003
Epoch 80: Loss for 'α': 0.004458 | Loss for 'β': 0.000004
Epoch 90: Loss for 'α': 0.006103 | Loss for 'β': 0.000004
Epoch 100: Loss for 'α': 0.005583 | Loss for 'β': 0.000007


## Checking for random values of A, B and $\kappa$

In [12]:
A, B, kappa = 0.6, 0.3, 0.25
t = np.linspace(50, 100, N)
z = mirror_trajectory(t, A, B, kappa)

alpha_true, beta_true = compute_alpha_beta(A, B, kappa)

with torch.no_grad():
    zt = torch.tensor(z, dtype=torch.complex64).unsqueeze(0).to(device)
    alpha_pred = alpha_net(zt)[0].cpu().numpy()
    beta_pred  = beta_net(zt)[0].cpu().numpy()

In [13]:
alpha_pred - alpha_true

array([[ 1.04+0.26j, -0.12-0.28j, -0.16+0.15j, -0.02+0.25j,  0.06+0.22j,
         0.08+0.16j,  0.06+0.09j,  0.05+0.04j,  0.05-0.j  ,  0.06-0.03j],
       [-0.53-0.16j, -0.11-0.25j,  0.01+0.23j,  0.13-0.02j, -0.04-0.06j,
        -0.1 +0.04j, -0.07+0.09j, -0.02+0.09j,  0.02+0.04j,  0.04-0.j  ],
       [-0.39+0.39j,  0.24+0.12j, -0.11+0.09j,  0.06-0.08j, -0.11+0.02j,
         0.01+0.09j,  0.04-0.01j, -0.02-0.06j, -0.06-0.03j, -0.05+0.02j],
       [-0.35+0.24j, -0.09-0.18j, -0.  -0.08j,  0.12+0.03j, -0.08-0.02j,
         0.04+0.03j, -0.01-0.07j, -0.06+0.j  ,  0.  +0.04j,  0.03-0.01j],
       [-0.18-0.27j, -0.18+0.04j, -0.02+0.1j , -0.08+0.09j, -0.02-0.06j,
         0.04+0.05j,  0.  -0.06j, -0.03+0.02j,  0.04-0.01j, -0.01-0.05j],
       [ 0.35-0.07j, -0.08+0.09j,  0.07-0.06j,  0.05-0.09j,  0.06-0.01j,
        -0.04+0.04j,  0.03-0.04j, -0.02+0.03j,  0.05-0.01j, -0.01-0.03j],
       [-0.25+0.14j, -0.12+0.08j, -0.06-0.09j, -0.08+0.06j, -0.04+0.04j,
        -0.01-0.05j,  0.05+0.j  , -0.04-0.j  

In [14]:
np.abs(alpha_pred - alpha_true)

array([[1.07, 0.3 , 0.23, 0.25, 0.23, 0.17, 0.11, 0.06, 0.05, 0.06],
       [0.56, 0.27, 0.23, 0.13, 0.07, 0.1 , 0.11, 0.09, 0.05, 0.04],
       [0.55, 0.27, 0.14, 0.1 , 0.11, 0.09, 0.04, 0.06, 0.07, 0.05],
       [0.42, 0.2 , 0.08, 0.12, 0.08, 0.05, 0.07, 0.06, 0.04, 0.03],
       [0.32, 0.19, 0.11, 0.12, 0.06, 0.06, 0.06, 0.04, 0.04, 0.05],
       [0.36, 0.12, 0.09, 0.1 , 0.06, 0.06, 0.05, 0.03, 0.05, 0.03],
       [0.29, 0.15, 0.11, 0.1 , 0.05, 0.05, 0.05, 0.04, 0.04, 0.02],
       [0.29, 0.15, 0.1 , 0.08, 0.05, 0.06, 0.04, 0.04, 0.03, 0.03],
       [0.26, 0.15, 0.1 , 0.09, 0.07, 0.05, 0.03, 0.04, 0.04, 0.02],
       [0.29, 0.1 , 0.07, 0.06, 0.04, 0.02, 0.04, 0.04, 0.03, 0.02]])

In [15]:
beta_pred - beta_true

array([[-0.01+0.03j,  0.01-0.j  ,  0.01-0.01j, -0.  -0.j  , -0.01+0.j  ,
        -0.  +0.01j,  0.  +0.j  ,  0.  +0.j  ,  0.  +0.j  ,  0.  +0.j  ],
       [-0.  +0.j  , -0.  -0.j  , -0.  +0.j  , -0.  -0.j  , -0.  -0.j  ,
        -0.  +0.j  ,  0.  +0.j  ,  0.  -0.j  , -0.  -0.j  , -0.  -0.j  ],
       [ 0.  +0.j  , -0.  +0.j  ,  0.  -0.j  ,  0.  -0.j  , -0.  +0.j  ,
        -0.  +0.j  ,  0.  +0.j  ,  0.  +0.j  ,  0.  -0.j  , -0.  -0.j  ],
       [-0.  -0.j  , -0.  +0.j  , -0.  +0.j  ,  0.  -0.j  , -0.  -0.j  ,
        -0.  +0.j  ,  0.  -0.j  ,  0.  +0.j  ,  0.  -0.j  ,  0.  -0.j  ],
       [ 0.  -0.j  , -0.  -0.j  , -0.  -0.j  , -0.  +0.j  ,  0.  +0.j  ,
        -0.  +0.j  ,  0.  -0.j  , -0.  +0.j  ,  0.  -0.j  , -0.  -0.j  ],
       [-0.  -0.j  ,  0.  +0.j  ,  0.  -0.j  , -0.  -0.j  ,  0.  +0.j  ,
        -0.  +0.j  , -0.  -0.j  ,  0.  +0.j  , -0.  -0.j  , -0.  -0.j  ],
       [ 0.  -0.j  , -0.  -0.j  ,  0.  +0.j  ,  0.  +0.j  ,  0.  -0.j  ,
         0.  +0.j  , -0.  -0.j  ,  0.  -0.j  

In [16]:
np.abs(beta_pred - beta_true)

array([[0.04, 0.01, 0.01, 0.01, 0.01, 0.01, 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ]])

## Checking for constraint: $\alpha^{\ast} \alpha^{T} - \beta \beta^{\dagger} = I$

In [17]:
alpha_pred.conj() @ alpha_pred.T - beta_pred @ beta_pred.conj().T

array([[ 0.12-0.j  ,  0.01+0.j  ,  0.  +0.02j,  0.01+0.02j,  0.01+0.j  ,
         0.01-0.01j,  0.  -0.j  ,  0.  +0.j  , -0.  +0.j  , -0.02+0.02j],
       [ 0.01-0.j  ,  0.01-0.j  , -0.  +0.j  ,  0.  +0.j  ,  0.  +0.j  ,
        -0.  +0.j  , -0.  +0.j  , -0.  -0.j  ,  0.  +0.j  ,  0.  -0.j  ],
       [ 0.  -0.02j, -0.  -0.j  ,  0.01+0.j  ,  0.  -0.j  , -0.  -0.j  ,
        -0.  -0.j  , -0.  -0.j  , -0.  -0.j  ,  0.  -0.j  ,  0.  +0.j  ],
       [ 0.01-0.02j,  0.  -0.j  ,  0.  +0.j  ,  0.01-0.j  , -0.  -0.j  ,
        -0.  -0.j  , -0.  +0.j  ,  0.  -0.j  ,  0.  -0.j  ,  0.  +0.j  ],
       [ 0.01-0.j  ,  0.  -0.j  , -0.  +0.j  , -0.  +0.j  ,  0.01-0.j  ,
         0.  -0.j  ,  0.  +0.j  ,  0.  +0.j  , -0.  +0.j  , -0.  +0.j  ],
       [ 0.01+0.01j, -0.  -0.j  , -0.  +0.j  , -0.  +0.j  ,  0.  +0.j  ,
         0.01+0.j  , -0.  -0.j  ,  0.  +0.j  , -0.  -0.j  , -0.  +0.j  ],
       [ 0.  +0.j  , -0.  -0.j  , -0.  +0.j  , -0.  -0.j  ,  0.  -0.j  ,
        -0.  +0.j  ,  0.  +0.j  , -0.  -0.j  

In [18]:
np.abs(alpha_pred.conj() @ alpha_pred.T - beta_pred @ beta_pred.conj().T - np.eye(M))

array([[0.88, 0.01, 0.02, 0.02, 0.01, 0.01, 0.  , 0.  , 0.01, 0.02],
       [0.01, 0.99, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.02, 0.  , 0.99, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.01],
       [0.02, 0.  , 0.  , 0.99, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.01, 0.  , 0.  , 0.  , 0.99, 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.01, 0.  , 0.  , 0.  , 0.  , 0.99, 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  ],
       [0.02, 0.  , 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.99]])

## Notes
- Range of t is (50, 100)
- Grid size is kept short (10) for now as the alpha and beta terms have exponents which blow up for higher values of p and omega
- Ranges of $A, B$ and $\kappa$ are also small for now
- The NN works better for $\beta$ than for $\alpha$
- Constraint equation is almost satisfied